<div dir="rtl">

# الدرس 54: استخدام النماذج عبر واجهة برمجية

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### خادم مزيف للتجربة بدون مفتاح

</div>

In [ ]:
import json
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer

class FakeAPI(BaseHTTPRequestHandler):
    replies = []            # queue of (status, json) the server will answer with
    received = []           # every request it got

    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        FakeAPI.received.append({"path": self.path, "version": self.headers.get("anthropic-version"), "body": body})
        status, reply = FakeAPI.replies.pop(0)
        data = json.dumps(reply).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def log_message(self, *args):
        pass

server = HTTPServer(("127.0.0.1", 0), FakeAPI)
threading.Thread(target=server.serve_forever, daemon=True).start()
FAKE_URL = f"http://127.0.0.1:{server.server_port}"

def fake_text_reply(text, n_in, n_out):
    return 200, {"id": "msg_fake", "type": "message", "role": "assistant", "model": "fake",
                 "content": [{"type": "text", "text": text}], "stop_reason": "end_turn",
                 "stop_sequence": None, "usage": {"input_tokens": n_in, "output_tokens": n_out}}
print("fake API listening on", FAKE_URL)

<div dir="rtl">

### أول طلب

</div>

In [ ]:
%pip install -q -U anthropic

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key="fake-key", base_url=FAKE_URL, max_retries=0)
FakeAPI.replies.append(fake_text_reply("الانتباه يجعل كل كلمة تنظر إلى الكلمات المهمة لها في الجملة.", 31, 22))

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    system="أنت مدرّس صبور. أجب بالعربية، بجملة واحدة.",
    messages=[{"role": "user", "content": "ما الانتباه في الترانسفورمرز؟"}],
)
print("answer     :", response.content[0].text)
print("stop reason:", response.stop_reason, "| tokens in/out:", response.usage.input_tokens, response.usage.output_tokens)

sent = FakeAPI.received[-1]
print("sent to    :", sent["path"], "| API version header:", sent["version"])
print("request    :", json.dumps(sent["body"], ensure_ascii=False))

In [ ]:
import os
if os.environ.get("ANTHROPIC_API_KEY"):              # runs only when you have a real key
    real_client = anthropic.Anthropic()              # reads the key from the environment
    reply = real_client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=300,
                                        messages=[{"role": "user", "content": "ما الانتباه في الترانسفورمرز؟"}])
    print(reply.content[0].text)

<div dir="rtl">

### محادثة من جولتين

</div>

In [ ]:
history = [{"role": "user", "content": "اقترح اسماً لقطة سوداء."}]
FakeAPI.replies.append(fake_text_reply("ليل.", 18, 4))
reply = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=100, messages=history)

history.append({"role": "assistant", "content": reply.content[0].text})    # we keep the memory
history.append({"role": "user", "content": "ولماذا هذا الاسم؟"})
FakeAPI.replies.append(fake_text_reply("لأن لونها يشبه الليل.", 35, 9))
reply = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=100, messages=history)

print("messages sent in the second request:", len(FakeAPI.received[-1]["body"]["messages"]))
for m in FakeAPI.received[-1]["body"]["messages"]:
    print(f"  {m['role']:9}: {m['content']}")

<div dir="rtl">

### الأدوات: دورة كاملة

</div>

In [ ]:
weather_tool = {
    "name": "get_weather",
    "description": "Current weather for a city. Returns temperature in Celsius and conditions.",
    "input_schema": {"type": "object",
                     "properties": {"city": {"type": "string", "description": "City name in English"}},
                     "required": ["city"]},
}

def get_weather(city):                     # your real function: an API, a database...
    return {"city": city, "temp_c": 19, "conditions": "partly cloudy"}

question = [{"role": "user", "content": "هل أحتاج معطفاً في إسطنبول الآن؟"}]

# 1) the model asks for the tool
FakeAPI.replies.append((200, {"id": "msg_fake", "type": "message", "role": "assistant", "model": "fake",
    "content": [{"type": "tool_use", "id": "toolu_fake_1", "name": "get_weather", "input": {"city": "Istanbul"}}],
    "stop_reason": "tool_use", "stop_sequence": None, "usage": {"input_tokens": 120, "output_tokens": 40}}))
first = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=300, tools=[weather_tool], messages=question)
call = next(b for b in first.content if b.type == "tool_use")
print("stop reason:", first.stop_reason, "| tool:", call.name, "| input:", call.input)

# 2) we run it, and send the result back
result = get_weather(**call.input)
followup = question + [
    {"role": "assistant", "content": first.content},
    {"role": "user", "content": [{"type": "tool_result", "tool_use_id": call.id, "content": json.dumps(result)}]},
]
FakeAPI.replies.append(fake_text_reply("الحرارة 19 درجة مع غيوم: معطف خفيف يكفي.", 180, 25))
final = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=300, tools=[weather_tool], messages=followup)
print("final answer:", final.content[0].text)
print("last message we sent:", json.dumps(FakeAPI.received[-1]["body"]["messages"][-1], ensure_ascii=False))

<div dir="rtl">

### الأخطاء وإعادة المحاولة

</div>

In [ ]:
FakeAPI.replies.append((429, {"type": "error", "error": {"type": "rate_limit_error", "message": "Too many requests"}}))
try:
    client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=50,
                           messages=[{"role": "user", "content": "مرحباً"}])
except anthropic.RateLimitError as e:
    print("rate limited:", e.status_code)

<div dir="rtl">

### البث المتدفق

</div>

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):              # our fake server does not stream: needs the real API
    with real_client.messages.stream(model="claude-haiku-4-5-20251001", max_tokens=500,
                                     messages=[{"role": "user", "content": "اكتب فقرة عن إسطنبول."}]) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)

<div dir="rtl">

## تمرين: محادثة أرخص

محادثة من عشرين جولة: المستخدم يكتب 150 رمزاً في كل جولة، والمساعد يرد بـ 300. والأسعار الافتراضية: 3 و 15 دولاراً لكل مليون.

1. اكتب قانون رموز الطلب في الجولة $t$، واحسب مجموع رموز الطلب للجولات العشرين.
2. احسب التكلفة الكلية.
3. فكرة للتوفير: في كل طلب، أرسل آخر أربع جولات كاملة فقط. احسب مجموع رموز الطلب الجديد، ونسبة التوفير.

</div>

In [ ]:
# اكتب حلّك هنا
